# Partea a 4-a. Transformer Contrastive infoNCE

In [ ]:
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch.nn as nn
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score
import torch
import numpy as np
import random

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from collections import Counter
import unicodedata
from torch.utils.data import Sampler
import torch.nn.functional as F

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

Si aici vom folosi functii de clean , de asemenea, set seed functioneaza doar daca raman aceiasi paramterii la model inclusiv acelasi numar de epoci.
- Trebuie cautat daca am weights diferite pe masini diferite

In [2]:
# FUNCTII UTILE le definim sus
def drop_columns(df, columns_to_drop):
    return df.drop(columns=columns_to_drop)

# avem o problema la moldova
# diacriticele aparent nu sunt unicode?
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)

    # remove combining chars, deci t de la tap nu va mai avea caciulita sepaarata va fi un singur char
    text = ''.join(c for c in text if not unicodedata.combining(c))

    text = text.replace("„", '"').replace("”", '"')
    text = text.replace("’", "'")

    # remove control chars
    text = ''.join(c for c in text if unicodedata.category(c)[0] != 'C')

    text = text.replace("\n", " ")
    text = text.replace('""', '"')
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def count_unk(texts, tokenizer, n=100):
    total_unk = 0
    total_tokens = 0

    for t in texts[:n]:
        enc = tokenizer(
            t,
            truncation=True,
            max_length=512
        )

        input_ids = enc["input_ids"]
        total_tokens += len(input_ids)

        unk_id = tokenizer.unk_token_id
        total_unk += sum(1 for x in input_ids if x == unk_id)

    return total_unk / total_tokens


def count_unk_absolute(texts, tokenizer, n=100):
    total_unk = 0
    for t in texts[:n]:
        enc = tokenizer(t, truncation=True, max_length=512)
        unk_id = tokenizer.unk_token_id
        total_unk += sum(1 for x in enc["input_ids"] if x == unk_id)
    return total_unk


def count_combining(texts, n=1000):
    count = 0
    for t in texts[:n]:
        if any(unicodedata.combining(c) for c in t):
            count += 1
    return count


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### Checkpoint Implementare

1. Contrastive Dataset
   - in lucrarea lui Huerta (generating user embeddings with transformers) se foloseste infoNCE ca functie de loss
   - ideea lor este sa faca 2 matrice , produs scalar, normalizare , aplicat info nce minimizat info nce, care se uita la diagonala principala si trebuie sa aiba valoare cea mai mare la prima pozitie (a autorului pozitiv)
   - pentru fiecare autor vom tine o lista de ferestre de 512 tokeni (pt bert) si de aici vom extage primele 400 pentru ca vrem 200 de batch-uri x 2 texte/autor
3.

Pas 1: pastram Author Dataset pentru setul de test pentru ca acolo nu mai avem perechi de autori ci doar identifivam direct dupa document

In [3]:
class AuthorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

Pas 2: Pentru validare si train ne trebuie o functie care mapeaza autorii la 2 ferestre (a se vedea matricea X si X_t). Fiecare text al unui autor este separat in ferestre de overlap = 0.1 (10%) salvez aceste ferestre intr-o lista. Pare redundant dar pentru robustete ma folosesc de tokenizer si decodez ca sa le am ca text direct nu ca encoding. `prepare_texts_wit-windows` tine cont de toate ferestrele si toate labels , fara a fi nevoie sa folosesc dicitionar

In [4]:
# definim o functie in care mapam fiecare autor la ferestre de 512 tokeni
def split_into_windows(text, tokenizer, max_length=512, overlap=0.1):
    # permitem si overlap
    # returnam o lista de stringuri (ferestre)
    stride = int(max_length * (1-overlap)) # cu asta parcurg ~ step
    # scoatem tokens cu tokenizer
    tokens = tokenizer(
        text,
        truncation=False, # pt windows
        add_special_tokens=False,
        return_tensors='pt' # pytorch tensors buni pt cuda
    )
    input_ids = tokens['input_ids'][0]

    # daca e mai mic ca max_len il returnam direct asa
    if len(input_ids) <= max_length:
        return [text]
    # impartim in ferestre
    windows = []
    for start in range(0, len(input_ids), stride):
        # iau end
        end = min(len(input_ids), start + max_length)
        window_ids = input_ids[start:end]
        # decodam pt compatibilitate cu ce o sa faca bert
        window_text = tokenizer.decode(window_ids, skip_special_tokens=True)
        windows.append(window_text)
        if end == len(input_ids):
            break

    return windows


def prepare_texts_with_windows(texts, labels, tokenizer, max_length=512,overlap=0.1):
    # acum mapam autori -> [win1, win2 ...]
    # returnam liste extinse de texte si etichete
    all_texts = []
    all_labels = []

    for text, label in zip(texts, labels):
        windows = split_into_windows(text, tokenizer, max_length, overlap)
        all_texts.extend(windows)
        all_labels.extend([label] * len(windows))

    counts = Counter(all_labels)
    print(f"Texte originale: {len(texts)}")
    print(f"Ferestre totale: {len(all_texts)}")
    for label, count in sorted(counts.items()):
        print(f"  autor {label}: {count} ferestre")

    return all_texts, all_labels


Pas 3: `ContrastiveDataset` returneaza garantat o pereche de texte pt acelasi autor (sau ferestre). Ca sa putem sa scoatem de la toti autorii in `get_item` folosim un index pentru label (faptul ca luam pe top 20 si top 40 ajuta pentru ca traducem direct eticheta autorului in index)

Traducere: Eu vreau asa: `[(A), (B), (C), ...]` unde fiecare element este de forma: `(text1_autor, text2_autor, autor)`. Deci 1 exemplu este : 2 texte (ferestre de text) din acelasi autor + label(autor).

Datasetul: `batch = [dataset[i], dataset[j], dataset[k]]` iar DataLoader poate sa ia : (A_pair1), (A_pair2), (A_pair3) , dar nu vrem asta ci vrem perechi din autori diferiti.
Prima varianta ar fi sa construim manual batch-urile: Pt fiecare autor: <br>
A → [pair1, pair2, pair3] <br>
B → [pair1, pair2, pair3] <br>
C → [pair1, pair2, pair3] <br>

O sa avem: <br>
batch0 = [A_pair1, B_pair1, C_pair1] <br>
batch1 = [A_pair2, B_pair2, C_pair2] <br>
batch2 = [A_pair3, B_pair3, C_pair3] <br>

Si datasetul devine: `dataset = [batch0, batch1, batch2]`, deci cand extrag `dataset[i]` am deja un batch.

Ne asiguram ca Dataset nu returneaza mereu aceasi pereche cu random seed generat la fiecare epoca `seed + epoch`


In [5]:
"""
initial am asa: texts = ["...", "...", "..."]
labels = [0, 0, 1, 2, 1, ...]

1 text = 1 exemplu
eu vreau: (text1, text2, label) = exemplu

Pas1: grupam pe autori Autor 0 = [win1, win2, ..]
"""
class ContrastiveAuthorDataset(Dataset):

  def __init__(self, texts, labels, batches_per_epoch=None, epoch=0, seed=42):

    # ne asiguram ca avem acelasi shuffle pe epoca
    rng = random.Random(seed + epoch)

    # grupam ferestrele pe autori
    # ne vom folosi de un dictionar pt asta
    # author_texts = { autor: [lista texte] }
    author_texts = {}
    for text, label in zip(texts, labels):
      # daca nu il am deja in dex il instantiez
      if label not in author_texts:
        author_texts[label] = []
      author_texts[label].append(text)

    #ok dar daca nu am macar 2 ferestre la un autor nu il pot pune pe ala
    self.valid_authors = sorted([
        a for a, t in author_texts.items() if len(t) >= 2
    ])

    # ok acum putem sa cream perechile pe autor
    # 1. copiezi lista de texte
    # 2. faci shuffle
    # 3. creezi perechi consecutive:
    #   (t1,t2), (t3,t4)
    # salvam ca:
    # author_pairs = {autor: [(t1,t2), (t3,t4)}

    # dar trebuie calculat minimul de ferestre
    min_windows = min(
        len(author_texts[a]) for a in self.valid_authors
    )

    # putem configura doar un anumit nr de perechi per autor
    # conditionat de numarul minim de ferestre
    # pe care il impartim la 2 ca luam perechi
    max_pairs = min_windows // 2

    if batches_per_epoch is None:
      n_pairs = max_pairs
    else:
      n_pairs = min(batches_per_epoch, max_pairs)

    # nu stiu cat de relevanta e asta sincer:
    # n_pairs = min(batches_per_epoch, max_perechi)
    print(f"Autori: {len(self.valid_authors)}")
    print(f"Min ferestre per autor: {min_windows}")
    print(f"Perechi per autor: {max_pairs}")

    # ok acum ca stim max_pairs trebuie sa generam fix atatea per autor
    # pentru fiecare index k :
    #   batch_k = []
    # pt fiecare autor:
    # iau perechea k  si adaug in batch
    self.pairs = []       #(text1, text2)
    self.pair_labels = [] # (label)

    for author in self.valid_authors:
      windows = author_texts[author].copy()
      rng.shuffle(windows) # riksyy
      # acum iau perechile consecutive dupa shuffle
      for i in range(0, n_pairs * 2, 2): # daca e impar min_windows??
        self.pairs.append((windows[i], windows[i+1]))
        self.pair_labels.append(author)

    print(f"Total perechi: {len(self.pairs)}")


  def __len__(self):
    return len(self.pairs)


  def __getitem__(self, index):
    # scoatem de la index exemplul
    text1, text2 = self.pairs[index]
    label = self.pair_labels[index]

    return {
        'text1': text1,
        'text2': text2,
        'label': label
    }

Deocamdata ave lista de perechi
Dar vrem batch!
Pas 4: Sampler <br>
Ca sa putem garanta ca un autor nu apare de 2 ori in batch tinem o lista de indecsi in care apare fiecare autor.
Cum transform: `[(A), (A), (B), (C), (B)]` in `batch = [A, B, C]`


In [6]:
class UniqueAuthorBatchSampler(Sampler):
  def __init__(self, dataset, seed=42):
    # grupam indici pe autor
    #   author_indices = {}

    # pentru fiecare index i:
    #   label = pair_labels[i]
    #   author_indices[label].append(i)

    # obtinem: A = [0,1], B = [2,4], C = [3,5]
    self.seed = seed

    self.author_indices = {}
    for idx, label in enumerate(dataset.pair_labels):
      if label not in self.author_indices:
        self.author_indices[label] = []
      self.author_indices[label].append(idx)

    # ok cate batch-uri putem avea?
    # as zice nr minim de exemple autor
    self.authors = sorted(self.author_indices.keys())
    self.batch_size = len(self.authors)

    self.n_batches = min(
      len(idxs) for idxs in self.author_indices.values()
    )
    print(f"Batch size: {self.batch_size}")
    print(f"Batches per epoca: {self.n_batches}")

  def __iter__(self):
    rng = random.Random(self.seed)

    # shuffle indicii per autor
    # pentru fiecare autor:
    #   amesteci lista de indici (pozitiile unde apare un autor)
    shuffled = {
        a: rng.sample(idxs, len(idxs))
        for a, idxs in self.author_indices.items()
    }
    # tranaformam indicii in iteratori
    iters = {a: iter(idxs) for a, idxs in shuffled.items()}
    for _ in range(self.n_batches):
      batch = []
      # acum shuffle oridne autori  in batch
      authors_shuffled = self.authors.copy()
      rng.shuffle(authors_shuffled)

      for author in authors_shuffled:
        batch.append(next(iters[author]))

      yield batch


  def __len__(self):
    return self.n_batches

In [7]:
# def build_contrastive_dataloaders(train_df, val_df, test_df,
#                                    tokenizer,
#                                    batches_per_epoch=200,
#                                    seed=42):
#     le = LabelEncoder()
#     le.fit(train_df['author'])

#     y_train = le.transform(train_df['author'])
#     y_val   = le.transform(val_df['author'])
#     y_test  = le.transform(test_df['author'])

#     texts_train = train_df['text'].apply(clean_text).tolist()
#     texts_val   = val_df['text'].apply(clean_text).tolist()
#     texts_test  = test_df['text'].apply(clean_text).tolist()

#     # test loader normal, restul facem altceva
#     test_ds     = AuthorDataset(texts_test, y_test, tokenizer)
#     test_loader = DataLoader(
#         test_ds, batch_size=8,
#         shuffle=False, num_workers=4, pin_memory=True
#     )

#     return (le, texts_train, y_train,
#             texts_val, y_val,
#             test_loader, texts_test, y_test)
def build_contrastive_dataloaders(train_df, val_df, test_df,
                                   tokenizer,
                                   batches_per_epoch=None,
                                   seed=42):
    le = LabelEncoder()
    le.fit(train_df['author'])

    y_train = le.transform(train_df['author'])
    y_val   = le.transform(val_df['author'])
    y_test  = le.transform(test_df['author'])

    texts_train = train_df['text'].apply(clean_text).tolist()
    texts_val   = val_df['text'].apply(clean_text).tolist()
    texts_test  = test_df['text'].apply(clean_text).tolist()

    # ferestre
    texts_train_w, y_train_w = prepare_texts_with_windows(
        texts_train, y_train, tokenizer
    )
    texts_val_w, y_val_w = prepare_texts_with_windows(
        texts_val, y_val, tokenizer
    )

    # test loader normal
    test_ds     = AuthorDataset(texts_test, y_test, tokenizer)
    test_loader = DataLoader(
        test_ds, batch_size=8,
        shuffle=False, num_workers=4, pin_memory=True
    )

    return (le,
            texts_train_w, y_train_w,
            texts_val_w,   y_val_w,
            texts_test,    y_test,
            test_loader)

In [8]:
def contrastive_collate_fn(batch):
  """
  Avem un batch de forma:
  [
  {"text1": "...", "text2": "...", "label": 0},
  {"text1": "...", "text2": "...", "label": 1},
  ...
  ]

  Cum il facem tensor?
  Trebuie sa extragem textele le bagam in lista si le tokenizam = input_ids + attention_mask (adica le face numere :)
  Asa acum trebuie sa cream view1 si view2
  1. ia batch (listă de dicționare)
 2. extrage:
   - texts1
   - texts2
   - labels
 3. tokenizează texts1
 4. tokenizează texts2
 5. convertește labels în tensor
 6. returnează totul organizat
  """
  # extragem
  #batch = batch[0]
  texts1 = [item['text1'] for item in batch]
  texts2 = [item['text2'] for item in batch]
  labels = [item['label'] for item in batch]

  enc1 = tokenizer(
      texts1, max_length=512,
      truncation=True, padding='max_length',
      return_tensors='pt'
  )

  enc2 =  tokenizer(
      texts2, max_length=512,
      truncation=True, padding='max_length',
      return_tensors='pt'
  )

  return {
      'input_ids1': enc1['input_ids'],
      'attention_mask_1': enc1['attention_mask'],
      'input_ids2': enc2['input_ids'],
      'attention_mask_2': enc2['attention_mask'],
      'label': torch.tensor(labels, dtype=torch.long) # se supara CE daca nu e long
  }



- vom folosi functia `make_loaders_per_epoch` in train

In [ ]:
def make_loaders_for_epoch(texts_train_w, y_train_w,
                            texts_val_w,   y_val_w,
                            batches_per_epoch, epoch, seed):
    train_ds = ContrastiveAuthorDataset(
        texts_train_w, y_train_w,
        batches_per_epoch=batches_per_epoch,
        epoch=epoch, seed=seed
    )
    val_ds = ContrastiveAuthorDataset(
        texts_val_w, y_val_w,
        batches_per_epoch=batches_per_epoch,
        epoch=epoch, seed=seed
    )

    train_sampler = UniqueAuthorBatchSampler(train_ds, seed=seed+epoch)
    val_sampler   = UniqueAuthorBatchSampler(val_ds,   seed=seed+epoch)

    train_loader = DataLoader(
        train_ds,
        batch_sampler=train_sampler,
        collate_fn=contrastive_collate_fn,
        num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_sampler=val_sampler,
        collate_fn=contrastive_collate_fn,
        num_workers=4, pin_memory=True
    )

    return train_loader, val_loader

### Functia de loss
- definim functia de loss infoNCE
- ce vrem de fapt de la functia asta?

Eu primesc un batch asa:
```
autor1 → (text1, text2)
autor2 → (text1, text2)
...
autorN → (text1, text2)
```

BERT scoate embeddinguri :

```
z1 = embeddings pentru text1  → (N, d)
z2 = embeddings pentru text2  → (N, d)
```

Vrem ca `z1[i]` sa fie apropiat de `z2[i]` (acelasi autor) si `z1[i]` să fie departe de `z2[j!=i]`  (alt autor)

- calculam cat de similara sunt matricele : `S = z1 @ z2^T`

```
        z2_1   z2_2   z2_3
z1_1     +      -      -
z1_2     -      +      -
z1_3     -      -      +
```

Diagonala trebuie sa aiba scor pozitiv la logits, apoi le bagam in softmax sa le transformam in probabilitati.

Temperatura ajuta sa determinam cat de sigur e modelul la predictii, adica putem sa dam o temperatura mai mica sa indepartam samples intre ele


In [10]:
def info_nce(z1, z2, temperature=0.07):
  # z1, z2 (N,d)  normalizate
  # similaritate (N,N)
  logits = z1 @ z2.T / temperature
  targets = torch.arange(z1.size(0), device=z1.device)

  loss_12 = F.cross_entropy(logits, targets)
  loss_21 = F.cross_entropy(logits.T, targets)

  return 0.5 * (loss_12 + loss_21), logits


# si loss total ce + lambda * info_nce
def compute_loss(logits, labels, embs_1, embs_2,
                 lambda_contrastive=0.3, temperature=0.07):
    loss_ce  = F.cross_entropy(logits, labels)
    loss_nce, _ = info_nce(embs_1, embs_2, temperature)
    loss     = loss_ce + lambda_contrastive * loss_nce
    return loss, loss_ce, loss_nce

### Modelul BERT fine tuned cu contrastive loss

In [ ]:
# facem mean pooling peste predictii
# media peste tokeni
def mean_pooling(bert_output, attention_mask):
  # scoatem embedding
  token_embeddings = bert_output.last_hidden_state

  # [1,1,1,0,0] → [[1],[1],[1],[0],[0]]
  # ca sa il pot inmulti cu embeddings
  # (N, seq_len) → (N, seq_len, 1)
  mask_expanded = attention_mask.unsqueeze(-1).float()
  sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim = 1)
  sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
  return sum_embeddings / sum_mask


class ContrastiveBERTAuthorship(nn.Module):
    def __init__(self, num_authors, projection_dim=128, dropout=0.2):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            "dumitrescustefan/bert-base-romanian-cased-v1"
        )

        # explicam aici de ce: nu mai salveaza activarile
        # recalculeaza la baxward
        # salvam memorie nu si timp
        # self.bert.gradient_checkpointing_enable()
        # totusi nu e bine aiic ca va avea inferenta mai lenta
        # il bagam in run contrastive
        self.dropout = nn.Dropout(dropout)

        # clasificare directa
        self.classifier = nn.Linear(768, num_authors)

        # pentru n-pair loss si user embeddings
        self.projection_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, projection_dim)
        )

    def forward(self, input_ids, attention_mask):
        bert_out   = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled     = mean_pooling(bert_out, attention_mask)
        pooled     = self.dropout(pooled)

        logits     = self.classifier(pooled)
        projection = self.projection_head(pooled)
        projection = torch.nn.functional.normalize(projection, p=2, dim=1)

        return logits, projection

    def get_embedding(self, input_ids, attention_mask):
        with torch.no_grad():
            bert_out   = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            pooled     = mean_pooling(bert_out, attention_mask)
            projection = self.projection_head(pooled)
            projection = torch.nn.functional.normalize(projection, p=2, dim=1)
        return projection


- extragere embedding la nivel de ferestre (util la modelul curent)

In [27]:
def extract_embeddings(model, loader, device):
    model.eval()
    all_embs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids1'].to(device)
            attention_mask = batch['attention_mask_1'].to(device)
            labels = batch['label'].to(device)

            _, embs = model(input_ids, attention_mask)

            all_embs.append(embs.cpu())
            all_labels.append(labels.cpu())

    return torch.cat(all_embs), torch.cat(all_labels)


def compute_similarity_stats(embs, labels):
    embs = torch.nn.functional.normalize(embs, dim=1)
    sim_matrix = embs @ embs.T

    labels = labels.numpy()
    sim_matrix = sim_matrix.numpy()

    same = []
    diff = []

    N = len(labels)
    for i in range(N):
        for j in range(i+1, N):
            if labels[i] == labels[j]:
                same.append(sim_matrix[i][j])
            else:
                diff.append(sim_matrix[i][j])

    return same, diff


def save_similarity_plot(same, diff, epoch, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    plt.figure()
    plt.hist(same, bins=50, alpha=0.5, label='same author')
    plt.hist(diff, bins=50, alpha=0.5, label='different author')
    plt.legend()
    plt.title(f"Similarity distribution epoch {epoch}")
    plt.savefig(f"{out_dir}/sim_epoch_{epoch}.png")
    plt.close()


def save_similarity_stats(same, diff, epoch, out_dir):
    import json
    stats = {
        "epoch": epoch,
        "same_mean": float(sum(same)/len(same)),
        "diff_mean": float(sum(diff)/len(diff))
    }

    with open(f"{out_dir}/stats_epoch_{epoch}.json", "w") as f:
        json.dump(stats, f, indent=2)

train

In [ ]:
def train_epoch_contrastive(model, loader, optimizer,
                              scheduler, device,
                              contrastive_weight=0.3, scaler=None):
    
    if scaler is None:
        scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    model.train()
    criterion_ce = nn.CrossEntropyLoss()
    total_loss   = 0.0
    total_ce     = 0.0
    total_npair  = 0.0

    for i, batch in enumerate(loader):
        input_ids_1      = batch['input_ids1'].to(device, non_blocking=True)
        attention_mask_1 = batch['attention_mask_1'].to(device, non_blocking=True)
        input_ids_2      = batch['input_ids2'].to(device, non_blocking=True)
        attention_mask_2 = batch['attention_mask_2'].to(device, non_blocking=True)
        labels           = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type):
            logits, embs_1 = model(input_ids_1, attention_mask_1)
            _,      embs_2 = model(input_ids_2, attention_mask_2)

            loss_ce    = criterion_ce(logits, labels)
            # loss_nce = info_nce(embs_1, embs_2)[0]  # [0] pentru ca returneaza (loss, logits)
            loss, loss_ce, loss_nce = compute_loss(logits, labels, embs_1, embs_2)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss  += loss.item()
        total_ce    += loss_ce.item()
        total_npair += loss_nce.item()

        if i % 100 == 0:
            print(f"  Batch {i}/{len(loader)} "
                  f"loss: {loss.item():.4f} "
                  f"CE: {loss_ce.item():.4f} "
                  f"Loss infoNCE: {loss_nce.item():.4f}")

    n = len(loader)
    return total_loss/n, total_ce/n, total_npair/n


def eval_epoch_contrastive(model, loader, device):
    model.eval()
    criterion_ce = nn.CrossEntropyLoss()
    total_loss   = 0.0
    all_preds    = []
    all_labels   = []

    with torch.no_grad():
        for batch in loader:
            input_ids_1      = batch['input_ids1'].to(device)
            attention_mask_1 = batch['attention_mask_1'].to(device)
            labels           = batch['label'].to(device)

            with torch.amp.autocast(device_type=device.type):
                logits, _ = model(input_ids_1, attention_mask_1)
                loss      = criterion_ce(logits, labels)
            
            preds     = torch.argmax(logits, dim=-1)

            total_loss += loss.item()
            all_preds  .extend(preds.cpu().numpy())
            all_labels .extend(labels.cpu().numpy())

    f1  = f1_score(all_labels, all_preds, average='macro')
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    return total_loss / len(loader), f1, acc

In [ ]:
def run_contrastive_experiment(train_df, val_df,
                                texts_train_w, y_train_w,
                                texts_val_w,   y_val_w,
                                num_labels, device,
                                dataset_name="",
                                epochs=6,
                                batches_per_epoch=None,
                                contrastive_weight=0.3,
                                seed=42):
    set_seed(seed)

    model = ContrastiveBERTAuthorship(num_authors=num_labels).to(device)
    model.bert.gradient_checkpointing_enable()  # doar la antrenare

    # estimam steps pentru scheduler
    temp_loader, _ = make_loaders_for_epoch(
        texts_train_w, y_train_w,
        texts_val_w,   y_val_w,
        batches_per_epoch, epoch=0, seed=seed
    )
    steps_per_epoch = len(temp_loader)
    del temp_loader

    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=steps_per_epoch // 10,
        num_training_steps=steps_per_epoch * epochs
    )

    best_f1    = 0
    model_path = f'best_contrastive_{dataset_name}.pt'

    train_losses, val_losses = [], []
    train_f1s,   val_f1s    = [], []
    train_accs,  val_accs   = [], []
    train_infonce           = []

    # scaler
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loader, val_loader = make_loaders_for_epoch(
            texts_train_w, y_train_w,
            texts_val_w,   y_val_w,
            batches_per_epoch, epoch=epoch, seed=seed
        )

        train_loss, ce, infonce = train_epoch_contrastive(
            model, train_loader, optimizer, scheduler,
            device, contrastive_weight, scaler=scaler
        )
        _, train_f1, train_acc = eval_epoch_contrastive(
            model, train_loader, device
        )
        val_loss, val_f1, val_acc = eval_epoch_contrastive(
            model, val_loader, device
        )

        embs, labels = extract_embeddings(model, val_loader, device)
        same, diff = compute_similarity_stats(embs, labels)

        save_similarity_plot(same, diff, epoch, "outputs")
        save_similarity_stats(same, diff, epoch, "outputs")

        train_losses.append(train_loss)
        val_losses  .append(val_loss)
        train_f1s   .append(train_f1)
        val_f1s     .append(val_f1)
        train_accs  .append(train_acc)
        val_accs    .append(val_acc)
        train_infonce.append(infonce)

        print(f"Train - loss: {train_loss:.4f} CE: {ce:.4f} "
              f"InfoNCE: {infonce:.4f} F1: {train_f1:.4f}")
        print(f"Val   - loss: {val_loss:.4f} F1: {val_f1:.4f} "
              f"Acc: {val_acc:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), model_path)
            print(f"  Salvat (F1={val_f1:.4f})")

    print(f"\nBest Val F1: {best_f1:.4f}")
    model.load_state_dict(torch.load(model_path, map_location=device))

    return model, model_path, {
        'train_losses': train_losses, 'val_losses':  val_losses,
        'train_f1s':    train_f1s,   'val_f1s':      val_f1s,
        'train_accs':   train_accs,  'val_accs':     val_accs,
        'train_infonce': train_infonce,'best_val_f1':  best_f1
    }

In [14]:
def predict_with_voting_contrastive(model, text, tokenizer, 
                                     device, max_length=512, overlap=0.1):
    stride = int(max_length * (1 - overlap))
    model.eval()
    
    tokens    = tokenizer(text, truncation=False,
                          add_special_tokens=True, return_tensors='pt')
    input_ids = tokens['input_ids'][0]
    
    predictions = []
    with torch.no_grad():
        for start in range(0, len(input_ids), stride):
            end    = min(start + max_length, len(input_ids))
            window = input_ids[start:end]
            
            if len(window) < max_length:
                padding = torch.zeros(max_length - len(window), dtype=torch.long)
                window  = torch.cat([window, padding])
            
            attn   = (window != 0).long()
            
            # diferenta fata de baseline — model returneaza (logits, projection)
            logits, _ = model(
                window.unsqueeze(0).to(device),
                attn.unsqueeze(0).to(device)
            )
            pred = torch.argmax(logits, dim=-1).item()
            predictions.append(pred)
            
            if end == len(input_ids):
                break
    
    return Counter(predictions).most_common(1)[0][0]

In [15]:
def build_user_embeddings(model, texts_train, y_train,
                           tokenizer, device, le,
                           max_length=512, batch_size=32):
    model.eval()
    author_embeddings = {}
    
    for author_idx, author_name in enumerate(le.classes_):
        # gasesti toate textele acestui autor
        indices    = [i for i, y in enumerate(y_train) if y == author_idx]
        autor_texts = [texts_train[i] for i in indices]
        
        embs = []
        for start in range(0, len(autor_texts), batch_size):
            batch_texts = autor_texts[start:start+batch_size]
            encoded = tokenizer(
                batch_texts, max_length=max_length,
                truncation=True, padding='max_length',
                return_tensors='pt'
            )
            emb = model.get_embedding(
                encoded['input_ids'].to(device),
                encoded['attention_mask'].to(device)
            )
            embs.append(emb.cpu())
        
        all_embs = torch.cat(embs, dim=0)
        author_embeddings[author_name] = torch.mean(all_embs, dim=0)
        print(f"  {author_name}: {len(indices)} texte")
    
    return author_embeddings


def predict_with_user_embeddings(model, text, author_embeddings,
                                  tokenizer, le, device, max_length=512):
    model.eval()
    encoded = tokenizer(
        text, max_length=max_length,
        truncation=True, padding='max_length',
        return_tensors='pt'
    )
    doc_emb = model.get_embedding(
        encoded['input_ids'].to(device),
        encoded['attention_mask'].to(device)
    ).squeeze(0).cpu()
    
    similarities = {
        author: torch.nn.functional.cosine_similarity(
            doc_emb.unsqueeze(0),
            emb.unsqueeze(0)
        ).item()
        for author, emb in author_embeddings.items()
    }
    
    pred_author = max(similarities, key=similarities.get)
    return le.transform([pred_author])[0]


def evaluate_contrastive(model, texts_test, y_test,
                          tokenizer, le, device,
                          dataset_name="", model_path=None,
                          author_embeddings=None):
    if model_path:
        model.load_state_dict(torch.load(model_path, map_location=device))
    
    print(f"\n--- Varianta A: clasificator + voting ---")
    preds_a = []
    for i, text in enumerate(texts_test):
        pred = predict_with_voting_contrastive(
            model, text, tokenizer, device
        )
        preds_a.append(pred)
        if i % 100 == 0:
            print(f"  {i}/{len(texts_test)}")
    
    f1_a  = f1_score(y_test, preds_a, average='macro')
    acc_a = (np.array(preds_a) == np.array(y_test)).mean()
    print(f"F1: {f1_a:.4f} | Acc: {acc_a:.4f}")
    print(classification_report(y_test, preds_a, target_names=le.classes_))
    
    # varianta B doar daca avem user embeddings
    f1_b, acc_b, preds_b = None, None, None
    if author_embeddings is not None:
        print(f"\n--- Varianta B: user embeddings + cosine ---")
        preds_b = []
        for i, text in enumerate(texts_test):
            pred = predict_with_user_embeddings(
                model, text, author_embeddings,
                tokenizer, le, device
            )
            preds_b.append(pred)
            if i % 100 == 0:
                print(f"  {i}/{len(texts_test)}")
        
        f1_b  = f1_score(y_test, preds_b, average='macro')
        acc_b = (np.array(preds_b) == np.array(y_test)).mean()
        print(f"F1: {f1_b:.4f} | Acc: {acc_b:.4f}")
        print(classification_report(y_test, preds_b, target_names=le.classes_))
    
    return preds_a, f1_a, acc_a, preds_b, f1_b, acc_b

#### Functie de testare corectitudine batch a nu se rula inutil

In [16]:
def test_sampler_and_loader(dataset, sampler_cls, collate_fn, tokenizer,
                            seed=42, num_batches_to_check=3):
    print("=== TEST SAMPLER ===")
    sampler = sampler_cls(dataset, seed=seed)

    # 1) Structura batch + unicitate autori
    for i, batch_indices in enumerate(sampler):
        print(f"\nBatch {i}: {batch_indices}")

        labels = [dataset.pair_labels[idx] for idx in batch_indices]
        print("Labels:", labels)

        # 1 autor / batch
        unique_labels = set(labels)
        print("Nr autori unici:", len(unique_labels))
        assert len(labels) == len(unique_labels), "[FAIL] Duplicate de autor în batch!"

        if i + 1 >= num_batches_to_check:
            break

    # 2) Reproductibilitate
    print("\n=== TEST REPRODUCIBILITATE ===")
    sampler1 = sampler_cls(dataset, seed=seed)
    sampler2 = sampler_cls(dataset, seed=seed)

    b1 = list(sampler1)[0]
    b2 = list(sampler2)[0]
    print("Same seed -> identic:", b1 == b2)
    assert b1 == b2, "[FAIL] Samplerul NU este reproductibil!"

    sampler3 = sampler_cls(dataset, seed=seed + 1)
    b3 = list(sampler3)[0]
    print("Seed diferit -> diferit:", b1 != b3)
    assert b1 != b3, "[FAIL] Seed diferit nu schimba ordinea!"

    # 3) DataLoader + collate
    print("\n=== TEST DATALOADER ===")
    loader = DataLoader(
        dataset,
        batch_sampler=sampler_cls(dataset, seed=seed),
        collate_fn=collate_fn
    )

    batch = next(iter(loader))

    print("Shape view1:", batch["input_ids1"].shape)
    print("Shape view2:", batch["input_ids2"].shape)

    labels = batch["label"]
    print("Labels tensor:", labels)
    print("Nr unice:", len(labels.unique()))

    # shape corect
    assert batch["input_ids1"].shape == batch["input_ids2"].shape, \
        "[FAIL] View-urile nu au aceeasi dimensiune!"

    # 1 autor per batch
    assert len(labels) == len(labels.unique()), \
        "[FAIL] Duplicate in labels după collate!"

    # 4) Nr batch-uri
    print("\n=== TEST NUMAR BATCH-URI ===")
    print("Nr batch-uri sampler:", len(sampler))

    print("\n[PASSED] TOATE TESTELE AU TRECUT")

In [17]:
def plot_training_curves(train_losses, val_losses,
                          train_f1s, val_f1s,
                          train_accs, val_accs,
                          dataset_name=""):
    epochs_range = range(1, len(train_losses) + 1)
    
    plt.figure(figsize=(18, 6))
    plt.suptitle(f'Training Progress — {dataset_name}', fontsize=14, y=1.02)
    
    # Loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, train_losses, label='Train Loss', marker='o', color='steelblue')
    plt.plot(epochs_range, val_losses,   label='Val Loss',   marker='o', color='coral')
    plt.title('Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # F1
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_f1s, label='Train F1', marker='o', color='blue')
    plt.plot(epochs_range, val_f1s,   label='Val F1',   marker='o', color='orange')
    plt.title('Macro F1 over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    # Accuracy
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, train_accs, label='Train Accuracy', marker='o', color='purple')
    plt.plot(epochs_range, val_accs,   label='Val Accuracy',   marker='o', color='green')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(f'training_curves_fusion_{dataset_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix_bert_fusion(y_true, y_pred, label_encoder,
                              dataset_name="", split='test'):

    labels = label_encoder.classes_
    n = len(labels)

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig_size = max(12, n * 0.8)

    font_size = max(6, int(300 / n))

    # === COUNTS ===
    plt.figure(figsize=(fig_size, fig_size))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        annot_kws={"size": font_size},
        square=True,
        cbar=True
    )

    plt.title(f'Confusion Matrix (Counts) — {dataset_name} | {split}', fontsize=16)
    plt.xlabel('Predicted')
    plt.ylabel('True')

    plt.xticks(rotation=45, ha='right', fontsize=font_size)
    plt.yticks(rotation=0, fontsize=font_size)

    plt.tight_layout()
    plt.savefig(f'cm_counts_fusion_{dataset_name}_{split}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # === NORMALIZED ===
    plt.figure(figsize=(fig_size, fig_size))
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        annot_kws={"size": font_size},
        square=True,
        cbar=True,
        vmin=0,
        vmax=1
    )

    plt.title(f'Confusion Matrix (Normalized) — {dataset_name} | {split}', fontsize=16)
    plt.xlabel('Predicted')
    plt.ylabel('True')

    plt.xticks(rotation=45, ha='right', fontsize=font_size)
    plt.yticks(rotation=0, fontsize=font_size)

    plt.tight_layout()
    plt.savefig(f'cm_norm_fusion_{dataset_name}_{split}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [18]:
tokenizer = AutoTokenizer.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


### Top 20 RO

In [20]:

top20_ro_train_text = pd.read_csv('final_stilo/top20_ro_train.csv')
top20_ro_val_text = pd.read_csv('final_stilo/top20_ro_val.csv')
top20_ro_test_text = pd.read_csv('final_stilo/top20_ro_test.csv')

In [21]:
le, texts_train_w, y_train_w, texts_val_w, y_val_w, texts_test, y_test, test_loader = build_contrastive_dataloaders(
    top20_ro_train_text,
    top20_ro_val_text,
    top20_ro_test_text,
    tokenizer
)

Texte originale: 49696
Ferestre totale: 66511
  autor 0: 3286 ferestre
  autor 1: 2492 ferestre
  autor 2: 4915 ferestre
  autor 3: 1550 ferestre
  autor 4: 2364 ferestre
  autor 5: 2169 ferestre
  autor 6: 2043 ferestre
  autor 7: 1453 ferestre
  autor 8: 5050 ferestre
  autor 9: 2033 ferestre
  autor 10: 4994 ferestre
  autor 11: 5997 ferestre
  autor 12: 2882 ferestre
  autor 13: 5855 ferestre
  autor 14: 5552 ferestre
  autor 15: 2491 ferestre
  autor 16: 1347 ferestre
  autor 17: 5246 ferestre
  autor 18: 3082 ferestre
  autor 19: 1710 ferestre
Texte originale: 1340
Ferestre totale: 1777
  autor 0: 105 ferestre
  autor 1: 54 ferestre
  autor 2: 136 ferestre
  autor 3: 43 ferestre
  autor 4: 63 ferestre
  autor 5: 55 ferestre
  autor 6: 48 ferestre
  autor 7: 26 ferestre
  autor 8: 159 ferestre
  autor 9: 50 ferestre
  autor 10: 142 ferestre
  autor 11: 161 ferestre
  autor 12: 76 ferestre
  autor 13: 149 ferestre
  autor 14: 146 ferestre
  autor 15: 62 ferestre
  autor 16: 35 fere

In [24]:
model, model_path, metrics = run_contrastive_experiment(
    train_df=top20_ro_train_text,
    val_df=top20_ro_val_text,
    texts_train_w=texts_train_w,
    y_train_w=y_train_w,
    texts_val_w=texts_val_w,
    y_val_w=y_val_w,
    num_labels=len(le.classes_),
    device=device
)

Autori: 20
Min ferestre per autor: 1347
Perechi per autor: 673
Total perechi: 13460
Autori: 20
Min ferestre per autor: 26
Perechi per autor: 13
Total perechi: 260
Batch size: 20
Batches per epoca: 673
Batch size: 20
Batches per epoca: 13

Epoch 1/3
Autori: 20
Min ferestre per autor: 1347
Perechi per autor: 673
Total perechi: 13460
Autori: 20
Min ferestre per autor: 26
Perechi per autor: 13
Total perechi: 260
Batch size: 20
Batches per epoca: 673
Batch size: 20
Batches per epoca: 13


OutOfMemoryError: CUDA out of memory. Tried to allocate 120.00 MiB. GPU 0 has a total capacity of 7.63 GiB of which 133.50 MiB is free. Including non-PyTorch memory, this process has 6.94 GiB memory in use. Of the allocated memory 6.72 GiB is allocated by PyTorch, and 105.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [17]:
le20, texts_train, y_train, texts_val, y_val, test_loader, texts_test, y_test = build_contrastive_dataloaders(top20_ro_train_text, top20_ro_val_text, top20_ro_test_text,
                                   tokenizer,
                                   batches_per_epoch=200,
                                   seed=42)

Texte originale: 49696
Ferestre totale: 66511
  autor 0: 3286 ferestre
  autor 1: 2492 ferestre
  autor 2: 4915 ferestre
  autor 3: 1550 ferestre
  autor 4: 2364 ferestre
  autor 5: 2169 ferestre
  autor 6: 2043 ferestre
  autor 7: 1453 ferestre
  autor 8: 5050 ferestre
  autor 9: 2033 ferestre
  autor 10: 4994 ferestre
  autor 11: 5997 ferestre
  autor 12: 2882 ferestre
  autor 13: 5855 ferestre
  autor 14: 5552 ferestre
  autor 15: 2491 ferestre
  autor 16: 1347 ferestre
  autor 17: 5246 ferestre
  autor 18: 3082 ferestre
  autor 19: 1710 ferestre
Texte originale: 1340
Ferestre totale: 1777
  autor 0: 105 ferestre
  autor 1: 54 ferestre
  autor 2: 136 ferestre
  autor 3: 43 ferestre
  autor 4: 63 ferestre
  autor 5: 55 ferestre
  autor 6: 48 ferestre
  autor 7: 26 ferestre
  autor 8: 159 ferestre
  autor 9: 50 ferestre
  autor 10: 142 ferestre
  autor 11: 161 ferestre
  autor 12: 76 ferestre
  autor 13: 149 ferestre
  autor 14: 146 ferestre
  autor 15: 62 ferestre
  autor 16: 35 fere

In [18]:
# check
print(f"Train ferestre: {len(texts_train)}, labels: {len(y_train)}")
print(f"Val ferestre:   {len(texts_val)}, labels: {len(y_val)}")
assert len(texts_train) == len(y_train)
assert len(texts_val)   == len(y_val)

Train ferestre: 66511, labels: 66511
Val ferestre:   1777, labels: 1777


Dataset returneaza liste de texte, dar pytorch are nevoie de batch de tensori nu texte, de aceea va trebui sa construim o functie `collate` care transforma in tensori

In [19]:
train_ds = ContrastiveAuthorDataset(
    texts_train,
    y_train,
    batches_per_epoch=200,
    epoch=0,
    seed=42
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=contrastive_collate_fn
)

Autori: 20
Min ferestre per autor: 1347
Perechi per autor: 673
Total perechi: 13460


In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_test = ContrastiveBERTAuthorship(num_authors=20).to(device)

batch = next(iter(train_loader))
logits, proj = model_test(
    batch['input_ids1'].to(device),
    batch['attention_mask_1'].to(device)
)
print(f"logits: {logits.shape}")    # (20, 20)
print(f"projection: {proj.shape}") # (20, 128)

logits: torch.Size([1, 20])
projection: torch.Size([1, 128])


In [51]:
test_sampler_and_loader(
    dataset=train_ds,
    sampler_cls=UniqueAuthorBatchSampler,
    collate_fn=contrastive_collate_fn,
    tokenizer=tokenizer
)

=== TEST SAMPLER ===
Batch size: 20
Batches per epoca: 673

Batch 0: [3892, 5647, 2260, 654, 9485, 12982, 11982, 10724, 8618, 6684, 1331, 8763, 6776, 4479, 4832, 1540, 2740, 12122, 7459, 11043]
Labels: [np.int64(5), np.int64(8), np.int64(3), np.int64(0), np.int64(14), np.int64(19), np.int64(17), np.int64(15), np.int64(12), np.int64(9), np.int64(1), np.int64(13), np.int64(10), np.int64(6), np.int64(7), np.int64(2), np.int64(4), np.int64(18), np.int64(11), np.int64(16)]
Nr autori unici: 20

Batch 1: [6346, 9857, 10982, 5577, 8050, 10161, 13164, 1471, 12479, 3649, 4707, 8730, 3044, 9287, 2679, 114, 7071, 11967, 1111, 5220]
Labels: [np.int64(9), np.int64(14), np.int64(16), np.int64(8), np.int64(11), np.int64(15), np.int64(19), np.int64(2), np.int64(18), np.int64(5), np.int64(6), np.int64(12), np.int64(4), np.int64(13), np.int64(3), np.int64(0), np.int64(10), np.int64(17), np.int64(1), np.int64(7)]
Nr autori unici: 20

Batch 2: [6387, 11175, 5380, 5389, 13337, 1811, 12403, 9194, 4206, 3742,

In [56]:
le, texts_train_w, y_train_w, texts_val_w, y_val_w, \
texts_test, y_test, test_loader = build_contrastive_dataloaders(
    top20_ro_train_text, top20_ro_val_text, top20_ro_test_text,
    tokenizer, batches_per_epoch=200, seed=42
)

train_loader, val_loader = make_loaders_for_epoch(
    texts_train_w, y_train_w,
    texts_val_w,   y_val_w,
    batches_per_epoch=200, epoch=0, seed=42
)

batch = next(iter(train_loader))
print(f"input_ids_1 shape: {batch['input_ids1'].shape}")   # (20, 512)
print(f"labels shape:      {batch['label'].shape}")         # (20,)
print(f"labels unice:      {len(batch['label'].unique())}") # 20

Texte originale: 49696
Ferestre totale: 66511
  autor 0: 3286 ferestre
  autor 1: 2492 ferestre
  autor 2: 4915 ferestre
  autor 3: 1550 ferestre
  autor 4: 2364 ferestre
  autor 5: 2169 ferestre
  autor 6: 2043 ferestre
  autor 7: 1453 ferestre
  autor 8: 5050 ferestre
  autor 9: 2033 ferestre
  autor 10: 4994 ferestre
  autor 11: 5997 ferestre
  autor 12: 2882 ferestre
  autor 13: 5855 ferestre
  autor 14: 5552 ferestre
  autor 15: 2491 ferestre
  autor 16: 1347 ferestre
  autor 17: 5246 ferestre
  autor 18: 3082 ferestre
  autor 19: 1710 ferestre
Texte originale: 1340
Ferestre totale: 1777
  autor 0: 105 ferestre
  autor 1: 54 ferestre
  autor 2: 136 ferestre
  autor 3: 43 ferestre
  autor 4: 63 ferestre
  autor 5: 55 ferestre
  autor 6: 48 ferestre
  autor 7: 26 ferestre
  autor 8: 159 ferestre
  autor 9: 50 ferestre
  autor 10: 142 ferestre
  autor 11: 161 ferestre
  autor 12: 76 ferestre
  autor 13: 149 ferestre
  autor 14: 146 ferestre
  autor 15: 62 ferestre
  autor 16: 35 fere

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


input_ids_1 shape: torch.Size([20, 512])
labels shape:      torch.Size([20])
labels unice:      20
